**Script name**: DWPSD

**Purpose**: Durbin-Watson Pro Simulation Dashboard (GUI): Up to 1 billion Monte Carlo Permutation Tests (MCPT) in Python

**Author**: Matteo Putzulu

**Version**: 5.9

**Date**: 11/5/2026

In [3]:
import time
import sys
import csv
import os
import io
import re
import math
import threading
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.lines import Line2D
from tabulate import tabulate
import tkinter as tk
from tkinter import filedialog, colorchooser, messagebox
import customtkinter as ctk

# =====================================================================
# CUSTOMTKINTER CONFIGURATION
# =====================================================================
ctk.set_appearance_mode("System")  # Options: "System", "Dark", "Light"
ctk.set_default_color_theme("blue")  # Options: "blue", "green", "dark-blue"

# =====================================================================
# UTILS & SCIENTIFIC NOTATION FOR FACTORIALS
# =====================================================================
def format_scientific(num):
    if num <= 0:
        return "0", 1
    val = math.factorial(num)
    total_digits = math.floor(math.log10(val)) + 1
    
    s = f"{val:.2e}"
    mantissa, exp = s.split('e')
    exp_int = int(exp)
    sup_digits = {'0':'⁰', '1':'¹', '2':'²', '3':'³', '4':'⁴', '5':'⁵', '6':'⁶', '7':'⁷', '8':'⁸', '9':'⁹', '-': '⁻'}
    exp_str = ''.join([sup_digits.get(c, c) for c in str(exp_int)])
    sci_str = f"{float(mantissa):.5f} ⋅ 10{exp_str}"
    return sci_str, total_digits

# =====================================================================
# CORE SIMULATION FUNCTION
# =====================================================================
def calculate_vectorized_dw(permutation_matrix):
    numerator = np.sum(np.diff(permutation_matrix, axis=1) ** 2, axis=1, dtype=np.float64)
    denominator = np.sum(permutation_matrix ** 2, axis=1, dtype=np.float64)
    denominator = np.where(denominator == 0, 1e-12, denominator)
    return numerator / denominator

def calculate_single_dw(values):
    values = np.asarray(values, dtype=np.float64)
    if len(values) < 2:
        return 0.0
    num = np.sum(np.diff(values) ** 2)
    den = np.sum(values ** 2)
    if den == 0:
        return 0.0
    return num / den

def run_simulation(values, target_scales, batch_size_val, progress_bar, eta_label, csv_filename, excel_filename, data_source_name, sample_seed_val, sim_seed_mode_val, sim_seed_val, update_gui_callback, check_stop_callback, sim_seed=None, use_exact_all=False):
    start_time = time.time()
    sample_size = len(values)
    values = np.asarray(values, dtype=np.float64)
    
    metrics = {"Runtime (s)": [], "Mean": [], "Median": [], "Min": [], "Max": [], "Q1": [], "Q3": [], "SD": []}
    dw_simulations_history = {}
    
    total_steps = len(target_scales)
    rng = np.random.default_rng(seed=sim_seed)
    
    for idx, n in enumerate(target_scales):
        if check_stop_callback():
            raise InterruptedError("Simulation stopped by user.")
            
        subset_start_time = time.time()
        progress_val = (idx / total_steps)
        
        if use_exact_all and n == math.factorial(sample_size):
            update_gui_callback(progress_val, f"Processing All exact permutations (n={n:,})...")
            all_perms_indices = np.array(list(itertools.permutations(range(sample_size))), dtype=np.int64)
            batch_size = batch_size_val
            num_perms = len(all_perms_indices)
            
            if num_perms <= batch_size:
                perm_matrix = values[all_perms_indices]
                dw_values = calculate_vectorized_dw(perm_matrix)
            else:
                dw_values = np.empty(num_perms, dtype=np.float64)
                num_batches = num_perms // batch_size
                remainder = num_perms % batch_size
                
                for b in range(num_batches):
                    if check_stop_callback():
                        raise InterruptedError("Simulation stopped by user.")
                    start_i = b * batch_size
                    end_i = start_i + batch_size
                    batch_indices = all_perms_indices[start_i:end_i]
                    dw_values[start_i:end_i] = calculate_vectorized_dw(values[batch_indices])
                    del batch_indices
                    
                    progress = (b + 1) / (num_batches + (1 if remainder > 0 else 0))
                    elapsed = time.time() - subset_start_time
                    eta = (elapsed / progress) - elapsed if progress < 1.0 else 0
                    eta_str = f"{int(eta // 60)}m {int(eta % 60)}s" if eta > 0 else "0s"
                    update_gui_callback(progress_val, f"Batch Progress (All n={n:,}): {int(progress*100)}% | ETA: {eta_str}")
                
                if remainder > 0:
                    if check_stop_callback():
                        raise InterruptedError("Simulation stopped by user.")
                    start_i = num_batches * batch_size
                    dw_values[start_i:] = calculate_vectorized_dw(values[all_perms_indices[start_i:]])
        else:
            update_gui_callback(progress_val, f"Processing scale n={n:,}...")
            if n <= 1_000_000:
                if check_stop_callback():
                    raise InterruptedError("Simulation stopped by user.")
                indices = rng.random((n, sample_size)).argsort(axis=1)
                perm_matrix = values[indices]
                dw_values = calculate_vectorized_dw(perm_matrix)
                del indices, perm_matrix
            else:
                batch_size = batch_size_val  
                num_batches = n // batch_size
                remainder = n % batch_size
                dw_values = np.empty(n, dtype=np.float64)
                
                for i in range(num_batches):
                    if check_stop_callback():
                        raise InterruptedError("Simulation stopped by user.")
                    indices = rng.random((batch_size, sample_size)).argsort(axis=1)
                    start_idx = i * batch_size
                    end_idx = start_idx + batch_size
                    dw_values[start_idx:end_idx] = calculate_vectorized_dw(values[indices])
                    del indices
                    
                    progress = (i + 1) / (num_batches + (1 if remainder > 0 else 0))
                    elapsed = time.time() - subset_start_time
                    eta = (elapsed / progress) - elapsed if progress < 1.0 else 0
                    eta_str = f"{int(eta // 60)}m {int(eta % 60)}s" if eta > 0 else "0s"
                    update_gui_callback(progress_val, f"Batch Progress (n={n:,}): {int(progress*100)}% | ETA: {eta_str}")
                    
                if remainder > 0:
                    if check_stop_callback():
                        raise InterruptedError("Simulation stopped by user.")
                    indices = rng.random((remainder, sample_size)).argsort(axis=1)
                    start_idx = num_batches * batch_size
                    dw_values[start_idx:] = calculate_vectorized_dw(values[indices])
                    del indices
        
        dw_simulations_history[n] = dw_values.copy()
        
        metrics["Runtime (s)"].append(f"{time.time() - subset_start_time:.1f}")
        metrics["Mean"].append(f"{np.mean(dw_values):.5f}")
        metrics["Median"].append(f"{np.median(dw_values):.5f}")
        metrics["Min"].append(f"{np.min(dw_values):.5f}")
        metrics["Max"].append(f"{np.max(dw_values):.5f}")
        metrics["Q1"].append(f"{np.percentile(dw_values, 25):.5f}")
        metrics["Q3"].append(f"{np.percentile(dw_values, 75):.5f}")
        metrics["SD"].append(f"{np.std(dw_values):.5f}")
        
    update_gui_callback(1.0, "Simulation Finished!")
    total_elapsed = time.time() - start_time
    
    formatted_headers = ["Index"] + [("All" if (use_exact_all and n == math.factorial(sample_size)) else (f"{n:,}" if n < 1_000_000 else (f"{n//1_000_000}M" if n < 1_000_000_000 else f"{n//1_000_000_000}B"))) for n in target_scales]
    table_rows = [[metric_name] + values_list for metric_name, values_list in metrics.items()]
    
    # Export CSV
    with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["# Data Source", data_source_name])
        writer.writerow(["# Sample seed", sample_seed_val])
        writer.writerow(["# Simulation seed mode", f"{sim_seed_mode_val} ({sim_seed_val})"])
        writer.writerow(["# Sample Values"] + [str(v) for v in values])
        writer.writerow([])
        writer.writerow(formatted_headers)
        writer.writerows(table_rows)
        
    # Export Excel (.xlsx)
    try:
        df_results = pd.DataFrame(table_rows, columns=formatted_headers)
        meta_info = pd.DataFrame([
            ["Data Source", data_source_name],
            ["Sample seed", sample_seed_val],
            ["Simulation seed mode", f"{sim_seed_mode_val} ({sim_seed_val})"],
            ["Sample Values", ", ".join([str(v) for v in values])]
        ], columns=["Parameter", "Value"])
        
        with pd.ExcelWriter(excel_filename, engine='openpyxl') as excel_writer:
            meta_info.to_excel(excel_writer, sheet_name='Metadata', index=False)
            df_results.to_excel(excel_writer, sheet_name='Simulation Results', index=False)
    except Exception as e:
        print(f"Warning: Could not save Excel file: {e}")
        
    return dw_simulations_history, table_rows, formatted_headers, total_elapsed

# =====================================================================
# MAIN APPLICATION WINDOW
# =====================================================================
class DurbinWatsonApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        
        self.title("Durbin-Watson Pro Simulation Dashboard")
        self.geometry("1100x1100")
        
        self.uploaded_file_path = None
        self.boxplot_color = "#a6c8e0"
        self.canvas_widget = None
        self.output_directory = ""
        self.stop_requested = False
        
        self.all_scale_var = tk.BooleanVar(value=False)
        self.all_scale_val = None
        self.center_mean_var = tk.BooleanVar(value=False)
        
        self.create_widgets()
        
    def create_widgets(self):
        self.main_container = ctk.CTkScrollableFrame(self, corner_radius=0, fg_color="transparent")
        self.main_container.pack(fill="both", expand=True, padx=10, pady=10)
        
        # --- HEADER ---
        header_frame = ctk.CTkFrame(self.main_container, fg_color="transparent")
        header_frame.pack(fill="x", pady=(0, 15))
        
        title_label = ctk.CTkLabel(header_frame, text="Durbin-Watson Pro Simulation Dashboard", font=ctk.CTkFont(size=20, weight="bold"))
        title_label.pack(side="left", anchor="w")
        
        right_header_frame = ctk.CTkFrame(header_frame, fg_color="transparent")
        right_header_frame.pack(side="right", anchor="e")
        
        link_label = ctk.CTkLabel(right_header_frame, text="github.com/eratosthenes-lab", font=ctk.CTkFont(size=12, weight="bold"), text_color="#2980b9", cursor="hand2")
        link_label.pack(anchor="e")
        link_label.bind("<Button-1>", lambda e: os.system("start https://github.com/eratosthenes-lab"))
        
        version_label = ctk.CTkLabel(right_header_frame, text="Version 5.9", font=ctk.CTkFont(size=11), text_color="gray")
        version_label.pack(anchor="e")
        
        # --- ROW 1: Data Source Configuration, Regenerate Button & Base DW / Mean / E[d] Statistic ---
        source_frame = ctk.CTkFrame(self.main_container)
        source_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(source_frame, text="Data Source:", width=120, anchor="w").pack(side="left", padx=10)
        self.data_source_combo = ctk.CTkComboBox(source_frame, values=['Random Normal', 'Manual Input', 'Upload File (CSV/Excel)'], command=self.on_source_change, width=200)
        self.data_source_combo.set('Random Normal')
        self.data_source_combo.pack(side="left", padx=10, pady=10)
        
        self.regenerate_btn = ctk.CTkButton(source_frame, text="Regenerate Random", command=self.regenerate_random_sample, width=150, fg_color="#8e44ad", hover_color="#732d91")
        self.regenerate_btn.pack(side="left", padx=10, pady=10)
        
        self.base_dw_label = ctk.CTkLabel(source_frame, text="Base Series DW: N/A | Mean: N/A | Expected E[d]: N/A", font=ctk.CTkFont(size=13, weight="bold"), text_color="#27ae60")
        self.base_dw_label.pack(side="left", padx=15, pady=10)
        
        # --- ROW 2: Sample Size (10 to 170, step 1), Seed & Center Mean Checkbox ---
        params_frame = ctk.CTkFrame(self.main_container)
        params_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(params_frame, text="Sample Size:", width=120, anchor="w").pack(side="left", padx=10)
        self.sample_size_slider = ctk.CTkSlider(params_frame, from_=10, to=170, number_of_steps=160, command=self.on_sample_size_change)
        self.sample_size_slider.set(50)
        self.sample_size_slider.pack(side="left", padx=10, fill="x", expand=True)
        
        self.sample_size_val_label = ctk.CTkLabel(params_frame, text="50", width=40)
        self.sample_size_val_label.pack(side="left", padx=5)
        
        ctk.CTkLabel(params_frame, text="Seed:", width=50).pack(side="left", padx=5)
        self.seed_entry = ctk.CTkEntry(params_frame, width=70)
        self.seed_entry.insert(0, "1")
        self.seed_entry.pack(side="left", padx=10, pady=10)
        self.seed_entry.bind("<KeyRelease>", self.update_permutations)
        
        self.center_mean_chk = ctk.CTkCheckBox(params_frame, text="Center Mean", variable=self.center_mean_var, command=self.update_permutations, fg_color="#2980b9")
        self.center_mean_chk.pack(side="left", padx=15, pady=10)
        
        # --- ROW 2.5: Simulation Seed Configuration ---
        sim_seed_frame = ctk.CTkFrame(self.main_container)
        sim_seed_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(sim_seed_frame, text="Simulation seed mode:", width=150, anchor="w").pack(side="left", padx=10)
        self.sim_seed_mode_combo = ctk.CTkComboBox(sim_seed_frame, values=['Non-Deterministic', 'Manual Seed'], command=self.on_sim_seed_mode_change, width=180)
        self.sim_seed_mode_combo.set('Non-Deterministic')
        self.sim_seed_mode_combo.pack(side="left", padx=10, pady=10)
        
        ctk.CTkLabel(sim_seed_frame, text="Simulation Seed:", width=120).pack(side="left", padx=5)
        self.sim_seed_entry = ctk.CTkEntry(sim_seed_frame, width=90)
        
        initial_entropy_seed = int(np.random.default_rng().integers(1, 1_000_000))
        self.sim_seed_entry.insert(0, str(initial_entropy_seed))
        self.sim_seed_entry.pack(side="left", padx=10, pady=10)
        self.sim_seed_entry.configure(state="disabled")
        
        self.regen_sim_seed_btn = ctk.CTkButton(sim_seed_frame, text="Regenerate Simulation Random", command=self.regenerate_sim_random, width=210, fg_color="#d35400", hover_color="#ba4a00")
        self.regen_sim_seed_btn.pack(side="left", padx=10, pady=10)
        
        # --- PERMUTATIONS INFO & DIGITS BOX ---
        perm_container = ctk.CTkFrame(self.main_container, fg_color="transparent")
        perm_container.pack(fill="x", padx=10, pady=(8, 2))
        
        self.perm_label = ctk.CTkLabel(perm_container, text="", font=ctk.CTkFont(size=16, weight="bold"), text_color="#2980b9")
        self.perm_label.pack(side="left", anchor="w")
        
        self.digits_box_frame = ctk.CTkFrame(perm_container, fg_color="#1f538d", corner_radius=6)
        self.digits_label = ctk.CTkLabel(self.digits_box_frame, text="Digits: 0", font=ctk.CTkFont(size=13, weight="bold"), text_color="white")
        self.digits_label.pack(padx=10, pady=4)
        
        # --- SAMPLE VALUES SCROLLABLE VIEWER ---
        sample_viewer_frame = ctk.CTkFrame(self.main_container, fg_color="transparent")
        sample_viewer_frame.pack(fill="x", padx=10, pady=(0, 8))
        
        ctk.CTkLabel(sample_viewer_frame, text="Extracted Sample Values:", font=ctk.CTkFont(size=13, weight="bold"), text_color="#7f8c8d").pack(anchor="w", pady=(0, 2))
        self.sample_textbox_view = ctk.CTkTextbox(sample_viewer_frame, height=75, font=ctk.CTkFont(family="Courier", size=14, weight="bold"))
        self.sample_textbox_view.pack(fill="x", expand=True)
        self.sample_textbox_view.configure(state="disabled")
        
        # --- ROW 3: Manual Input Area ---
        self.manual_frame = ctk.CTkFrame(self.main_container)
        self.manual_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(self.manual_frame, text="Manual Data:", width=120, anchor="nw").pack(side="left", padx=10, pady=10)
        self.manual_textbox = ctk.CTkTextbox(self.manual_frame, height=70)
        self.manual_textbox.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        self.manual_textbox.bind("<KeyRelease>", self.update_permutations)
        
        # --- ROW 4: File Upload Area & Column Selector ---
        self.upload_frame = ctk.CTkFrame(self.main_container)
        self.upload_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(self.upload_frame, text="Upload File:", width=120, anchor="w").pack(side="left", padx=10)
        self.upload_btn = ctk.CTkButton(self.upload_frame, text="Browse CSV/Excel...", command=self.browse_file, width=150)
        self.upload_btn.pack(side="left", padx=10, pady=10)
        
        self.file_status_label = ctk.CTkLabel(self.upload_frame, text="No file selected", text_color="gray", width=150, anchor="w")
        self.file_status_label.pack(side="left", padx=5)
        
        ctk.CTkLabel(self.upload_frame, text="Column:", width=60, anchor="w").pack(side="left", padx=(15, 5))
        self.column_combo = ctk.CTkComboBox(self.upload_frame, values=['Select file first'], command=self.on_column_change, width=160, state="disabled")
        self.column_combo.pack(side="left", padx=5, pady=10)
        
        self.toggle_source_inputs('Random Normal')
        
        # --- ROW 5: Export Settings & Batch Size ---
        export_frame = ctk.CTkFrame(self.main_container)
        export_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(export_frame, text="CSV Export:", width=80, anchor="w").pack(side="left", padx=(10, 2))
        self.csv_name_entry = ctk.CTkEntry(export_frame, width=130)
        self.csv_name_entry.insert(0, "dw_results.csv")
        self.csv_name_entry.pack(side="left", padx=2, pady=10)
        
        ctk.CTkLabel(export_frame, text="Excel Export:", width=85, anchor="w").pack(side="left", padx=(8, 2))
        self.excel_name_entry = ctk.CTkEntry(export_frame, width=130)
        self.excel_name_entry.insert(0, "dw_results.xlsx")
        self.excel_name_entry.pack(side="left", padx=2, pady=10)
        
        ctk.CTkLabel(export_frame, text="Image:", width=50, anchor="w").pack(side="left", padx=(8, 2))
        self.img_name_entry = ctk.CTkEntry(export_frame, width=110)
        self.img_name_entry.insert(0, "dw_boxplots.png")
        self.img_name_entry.pack(side="left", padx=2, pady=10)
        
        ctk.CTkLabel(export_frame, text="Format:", width=50, anchor="w").pack(side="left", padx=(5, 2))
        self.img_format_combo = ctk.CTkComboBox(export_frame, values=['png', 'jpg', 'pdf', 'svg'], width=70, command=self.update_image_extension)
        self.img_format_combo.set('png')
        self.img_format_combo.pack(side="left", padx=2, pady=10)
        
        ctk.CTkLabel(export_frame, text="Batch:", width=45, anchor="w").pack(side="left", padx=(8, 2))
        self.batch_size_combo = ctk.CTkComboBox(export_frame, values=['1M', '2M', '5M', '10M'], width=75)
        self.batch_size_combo.set('2M')
        self.batch_size_combo.pack(side="left", padx=2, pady=10)
        
        # --- ROW 5.5: Output Directory Selector ---
        dir_frame = ctk.CTkFrame(self.main_container)
        dir_frame.pack(fill="x", pady=5)
        
        ctk.CTkLabel(dir_frame, text="Save Directory:", width=130, anchor="w").pack(side="left", padx=10)
        self.dir_path_label = ctk.CTkLabel(dir_frame, text="Default (Current Directory)", text_color="gray", anchor="w")
        self.dir_path_label.pack(side="left", padx=10, fill="x", expand=True)
        
        self.dir_btn = ctk.CTkButton(dir_frame, text="Browse Folder...", command=self.browse_directory, width=140, fg_color="#2980b9")
        self.dir_btn.pack(side="right", padx=10, pady=10)
        
        # --- ROW 6: Color & Scales Selection ---
        settings_frame = ctk.CTkFrame(self.main_container)
        settings_frame.pack(fill="x", pady=5)
        
        color_btn = ctk.CTkButton(settings_frame, text="Choose Box Color", command=self.choose_color, fg_color="#2980b9")
        color_btn.pack(side="left", padx=15, pady=15)
        
        scales_frame = ctk.CTkFrame(settings_frame, fg_color="transparent")
        scales_frame.pack(side="left", fill="x", expand=True, padx=10, pady=10)
        
        ctk.CTkLabel(scales_frame, text="Scales (n) Selection:", anchor="w").pack(anchor="w", pady=(0, 5))
        
        self.scale_vars = {}
        self.scale_chks = {}  
        available_scales = [10, 100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000, 100_000_000, 1_000_000_000]
        default_scales = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
        
        self.scales_subframe = ctk.CTkFrame(scales_frame, fg_color="transparent")
        self.scales_subframe.pack(fill="x")
        
        for s in available_scales:
            var = tk.BooleanVar(value=(s in default_scales))
            self.scale_vars[s] = var
            lbl_text = f"{s:,}" if s < 1_000_000 else (f"{s//1_000_000}M" if s < 1_000_000_000 else f"{s//1_000_000_000}B")
            chk = ctk.CTkCheckBox(self.scales_subframe, text=lbl_text, variable=var, width=70)
            chk.pack(side="left", padx=1, pady=2)
            self.scale_chks[s] = chk  
            
        self.all_chk = ctk.CTkCheckBox(self.scales_subframe, text="All", variable=self.all_scale_var, width=110, fg_color="#e67e22")
        
        # --- ACTION BUTTONS & PROGRESS ---
        action_frame = ctk.CTkFrame(self.main_container)
        action_frame.pack(fill="x", pady=10)
        
        self.run_btn = ctk.CTkButton(action_frame, text="Launch Simulation", fg_color="#27ae60", hover_color="#219653", command=self.start_simulation_thread, height=40, font=ctk.CTkFont(size=15, weight="bold"))
        self.run_btn.pack(side="left", padx=10, pady=10)
        
        self.stop_btn = ctk.CTkButton(action_frame, text="Stop Simulation", fg_color="#d35400", hover_color="#ba4a00", command=self.stop_simulation, height=40, font=ctk.CTkFont(size=15, weight="bold"), state="disabled")
        self.stop_btn.pack(side="left", padx=10, pady=10)
        
        self.reset_btn = ctk.CTkButton(action_frame, text="Reset Data", fg_color="#c0392b", hover_color="#a93226", command=self.reset_application, height=40, font=ctk.CTkFont(size=15, weight="bold"))
        self.reset_btn.pack(side="left", padx=10, pady=10)
        
        progress_subframe = ctk.CTkFrame(action_frame, fg_color="transparent")
        progress_subframe.pack(side="left", fill="x", expand=True, padx=10)
        
        self.progress_bar = ctk.CTkProgressBar(progress_subframe)
        self.progress_bar.set(0)
        self.progress_bar.pack(fill="x", pady=(0, 5))
        
        self.eta_label = ctk.CTkLabel(progress_subframe, text="Status: Ready", anchor="w")
        self.eta_label.pack(fill="x")
        
        # --- RESULTS OUTPUT CONSOLE ---
        self.output_textbox = ctk.CTkTextbox(self.main_container, height=300, font=ctk.CTkFont(family="Courier", size=15, weight="bold"))
        self.output_textbox.pack(fill="both", expand=True, pady=10)
        
        # --- PLOT CONTAINER ---
        self.plot_frame = ctk.CTkFrame(self.main_container)
        self.plot_frame.pack(fill="both", expand=True, pady=10)
        
        ctk.CTkLabel(self.plot_frame, text="Distribution Boxplots (Interactive View)", font=ctk.CTkFont(size=14, weight="bold")).pack(anchor="w", padx=10, pady=5)
        
        self.update_permutations()

    def update_image_extension(self, fmt):
        current_name = os.path.splitext(self.img_name_entry.get())[0]
        self.img_name_entry.delete(0, "end")
        self.img_name_entry.insert(0, f"{current_name}.{fmt}")

    def stop_simulation(self):
        self.stop_requested = True
        self.eta_label.configure(text="Status: Stopping simulation...")

    def reset_application(self):
        self.stop_requested = True
        self.data_source_combo.set('Random Normal')
        self.sample_size_slider.set(50)
        self.sample_size_val_label.configure(text="50")
        self.seed_entry.delete(0, "end")
        self.seed_entry.insert(0, "1")
        self.center_mean_var.set(False)
        self.batch_size_combo.set('2M')
        
        self.sim_seed_mode_combo.set('Non-Deterministic')
        new_entropy_seed = int(np.random.default_rng().integers(1, 1_000_000))
        self.sim_seed_entry.configure(state="normal")
        self.sim_seed_entry.delete(0, "end")
        self.sim_seed_entry.insert(0, str(new_entropy_seed))
        self.sim_seed_entry.configure(state="disabled")
        self.regen_sim_seed_btn.configure(state="normal")
        
        self.manual_textbox.delete("0.1", "end")
        self.uploaded_file_path = None
        self.file_status_label.configure(text="No file selected", text_color="gray")
        self.column_combo.configure(values=['Select file first'], state="disabled")
        self.column_combo.set('Select file first')
        
        self.csv_name_entry.delete(0, "end")
        self.csv_name_entry.insert(0, "dw_results.csv")
        self.excel_name_entry.delete(0, "end")
        self.excel_name_entry.insert(0, "dw_results.xlsx")
        self.img_name_entry.delete(0, "end")
        self.img_name_entry.insert(0, "dw_boxplots.png")
        self.img_format_combo.set('png')
        self.output_directory = ""
        self.dir_path_label.configure(text="Default (Current Directory)", text_color="gray")
        
        default_scales = [10, 100, 1_000, 10_000, 100_000, 1_000_000]
        for s, var in self.scale_vars.items():
            var.set(s in default_scales)
        
        self.output_textbox.delete("0.1", "end")
        self.progress_bar.set(0)
        self.eta_label.configure(text="Status: Ready")
        
        if self.canvas_widget:
            self.canvas_widget.get_tk_widget().destroy()
            self.canvas_widget = None
            
        self.toggle_source_inputs('Random Normal')
        self.update_permutations()

    def on_sample_size_change(self, val):
        int_val = int(round(val))
        self.sample_size_val_label.configure(text=str(int_val))
        self.update_permutations()

    def on_source_change(self, source):
        self.toggle_source_inputs(source)
        self.update_permutations()

    def on_column_change(self, choice):
        self.update_permutations()

    def on_sim_seed_mode_change(self, mode):
        if mode == 'Manual Seed':
            self.sim_seed_entry.configure(state="normal")
            self.sim_seed_entry.delete(0, "end")
            self.sim_seed_entry.insert(0, "1")
            self.regen_sim_seed_btn.configure(state="disabled")
        else:
            new_entropy_seed = int(np.random.default_rng().integers(1, 1_000_000))
            self.sim_seed_entry.configure(state="normal")
            self.sim_seed_entry.delete(0, "end")
            self.sim_seed_entry.insert(0, str(new_entropy_seed))
            self.sim_seed_entry.configure(state="disabled")
            self.regen_sim_seed_btn.configure(state="normal")

    def regenerate_sim_random(self):
        if self.sim_seed_mode_combo.get() == 'Non-Deterministic':
            new_entropy_seed = int(np.random.default_rng().integers(1, 1_000_000))
            self.sim_seed_entry.configure(state="normal")
            self.sim_seed_entry.delete(0, "end")
            self.sim_seed_entry.insert(0, str(new_entropy_seed))
            self.sim_seed_entry.configure(state="disabled")

    def toggle_source_inputs(self, source):
        if source == 'Random Normal':
            self.manual_textbox.configure(state="disabled")
            self.upload_btn.configure(state="disabled")
            self.column_combo.configure(state="disabled")
            self.regenerate_btn.configure(state="normal")
        elif source == 'Manual Input':
            self.manual_textbox.configure(state="normal")
            self.upload_btn.configure(state="disabled")
            self.column_combo.configure(state="disabled")
            self.regenerate_btn.configure(state="disabled")
        elif source == 'Upload File (CSV/Excel)':
            self.manual_textbox.configure(state="disabled")
            self.upload_btn.configure(state="normal")
            if self.uploaded_file_path:
                self.column_combo.configure(state="normal")
            self.regenerate_btn.configure(state="disabled")

    def regenerate_random_sample(self):
        if self.data_source_combo.get() == 'Random Normal':
            new_seed = int(np.random.default_rng().integers(1, 1_000_000))
            self.seed_entry.delete(0, "end")
            self.seed_entry.insert(0, str(new_seed))
            self.update_permutations()

    def browse_file(self):
        filename = filedialog.askopenfilename(filetypes=[("CSV & Excel Files", "*.csv *.xls *.xlsx")])
        if filename:
            self.uploaded_file_path = filename
            self.file_status_label.configure(text=os.path.basename(filename), text_color="#27ae60")
            
            try:
                if filename.endswith('.csv'):
                    df = pd.read_csv(filename)
                else:
                    df = pd.read_excel(filename)
                
                numeric_cols = [str(col) for col in df.select_dtypes(include=[np.number]).columns.tolist()]
                if not numeric_cols:
                    numeric_cols = [str(col) for col in df.columns.tolist()] 
                
                if numeric_cols:
                    self.column_combo.configure(values=numeric_cols, state="normal")
                    self.column_combo.set(numeric_cols[0])
                else:
                    self.column_combo.configure(values=['No columns found'], state="disabled")
            except Exception as e:
                messagebox.showerror("File Error", f"Could not read file columns:\n{e}")
                return
                
            self.update_permutations()

    def update_permutations(self, event=None):
        source = self.data_source_combo.get()
        try:
            n = 0
            values = []
            if source == 'Random Normal':
                n = int(round(self.sample_size_slider.get()))
                seed_val = int(self.seed_entry.get()) if self.seed_entry.get().strip().isdigit() else 1
                rng = np.random.default_rng(seed=seed_val)
                values = rng.standard_normal(n)
                
            elif source == 'Manual Input':
                text = self.manual_textbox.get("0.1", "end").strip()
                if not text:
                    values = []
                else:
                    tokens = re.split(r'[,\s\n]+', text)
                    values = [float(t) for t in tokens if t]
                n = len(values)
                
                if n < 10 or n > 170:
                    msg = f"Elements: {n} (at least 10 numbers required)" if n < 10 else f"Elements: {n} (max 170 numbers allowed)"
                    self.perm_label.configure(text=msg, text_color="#e74c3c")
                    self.digits_box_frame.pack_forget()
                    self.display_sample_values([])
                    self.base_dw_label.configure(text="Base Series DW: N/A | Mean: N/A | Expected E[d]: N/A")
                    self.hide_all_checkbox()
                    self.disable_exceeding_scale_checkboxes(0)
                    return
                else:
                    values = np.array(values, dtype=np.float64)

            elif source == 'Upload File (CSV/Excel)':
                if self.uploaded_file_path:
                    if self.uploaded_file_path.endswith('.csv'):
                        df = pd.read_csv(self.uploaded_file_path)
                    else:
                        df = pd.read_excel(self.uploaded_file_path)
                    
                    selected_col = self.column_combo.get()
                    if selected_col and selected_col in df.columns:
                        values = pd.to_numeric(df[selected_col], errors='coerce').dropna().values
                    else:
                        numeric_df = df.select_dtypes(include=[np.number])
                        values = numeric_df.iloc[:, 0].dropna().values if not numeric_df.empty else np.array([])
                    n = len(values)
                else:
                    values = np.array([], dtype=np.float64)
                    n = 0

                if n < 10 or n > 170:
                    msg = f"Elements: {n} (at least 10 numbers required)" if n < 10 else f"Elements: {n} (max 170 numbers allowed)"
                    self.perm_label.configure(text=msg, text_color="#e74c3c")
                    self.digits_box_frame.pack_forget()
                    self.display_sample_values(values if n > 0 else [])
                    self.base_dw_label.configure(text="Base Series DW: N/A | Mean: N/A | Expected E[d]: N/A")
                    self.hide_all_checkbox()
                    self.disable_exceeding_scale_checkboxes(0)
                    return
                else:
                    values = np.array(values, dtype=np.float64)

            # Apply centering if checked
            if len(values) > 0 and self.center_mean_var.get():
                values = values - np.mean(values)

            # Display values, compute base DW, Mean & Theoretical Expected Value E[d]
            if len(values) >= 2:
                self.display_sample_values(values)
                base_dw = calculate_single_dw(values)
                sample_mean = np.mean(values)
                sum_sq = np.sum(values ** 2)
                
                if sum_sq > 0 and n > 1:
                    e_d = (2 * n / (n - 1)) * (1 - (n * (sample_mean ** 2)) / sum_sq)
                else:
                    e_d = 2.0
                    
                self.base_dw_label.configure(text=f"Base Series DW: {base_dw:.5f} | Mean: {sample_mean:.5f} | Expected E[d]: {e_d:.5f}")
            else:
                self.display_sample_values(values if len(values) > 0 else [])
                self.base_dw_label.configure(text="Base Series DW: N/A | Mean: N/A | Expected E[d]: N/A")

            if n < 2 or n > 170:
                self.perm_label.configure(text=f"Elements: {n} (must be between 10 and 170)", text_color="#e74c3c")
                self.digits_box_frame.pack_forget()
                self.hide_all_checkbox()
                self.disable_exceeding_scale_checkboxes(0)
            else:
                total_perms = math.factorial(n)
                perm_str, total_digits = format_scientific(n)
                self.perm_label.configure(text=f"Possible Permutations ({n}!): {perm_str}", text_color="#2980b9")
                self.digits_label.configure(text=f"Digits: {total_digits}")
                self.digits_box_frame.pack(side="left", padx=15, pady=2)
                
                self.disable_exceeding_scale_checkboxes(total_perms)

                # PERMETTI "ALL" ESCLUSIVAMENTE PER N = 10, 11, 12
                if n in (10, 11, 12):
                    self.all_scale_val = total_perms
                    self.all_chk.configure(text=f"All ({total_perms:,})")
                    self.all_chk.pack(side="left", padx=5, pady=2)
                else:
                    self.hide_all_checkbox()
        except Exception:
            self.perm_label.configure(text="Error calculating permutations", text_color="#e74c3c")
            self.digits_box_frame.pack_forget()
            self.display_sample_values([])
            self.base_dw_label.configure(text="Base Series DW: N/A | Mean: N/A | Expected E[d]: N/A")
            self.hide_all_checkbox()
            self.disable_exceeding_scale_checkboxes(0)

    def disable_exceeding_scale_checkboxes(self, total_perms):
        for s, chk in self.scale_chks.items():
            if total_perms > 0 and s > total_perms:
                chk.configure(state="disabled")
                self.scale_vars[s].set(False)
            else:
                chk.configure(state="normal")

    def display_sample_values(self, values):
        self.sample_textbox_view.configure(state="normal")
        self.sample_textbox_view.delete("0.1", "end")
        if len(values) > 0:
            formatted_str = ", ".join([f"{v:.4f}" if isinstance(v, (float, np.floating)) else str(v) for v in values])
            self.sample_textbox_view.insert("0.1", formatted_str)
        else:
            self.sample_textbox_view.insert("0.1", "No sample data available.")
        self.sample_textbox_view.configure(state="disabled")

    def hide_all_checkbox(self):
        self.all_scale_var.set(False)
        self.all_scale_val = None
        self.all_chk.pack_forget()

    def browse_directory(self):
        directory = filedialog.askdirectory(title="Select Output Directory")
        if directory:
            self.output_directory = directory
            self.dir_path_label.configure(text=directory, text_color="#27ae60")

    def choose_color(self):
        color = colorchooser.askcolor(title="Choose Boxplot Color", color=self.boxplot_color)
        if color[1]:
            self.boxplot_color = color[1]

    def get_input_values(self):
        source = self.data_source_combo.get()
        if source == 'Random Normal':
            seed_val = int(self.seed_entry.get()) if self.seed_entry.get().strip().isdigit() else 1
            rng = np.random.default_rng(seed=seed_val)
            values = rng.standard_normal(int(round(self.sample_size_slider.get())))
        
        elif source == 'Manual Input':
            text = self.manual_textbox.get("0.1", "end").strip()
            if not text:
                raise ValueError("Manual data field is empty. Please enter numbers.")
            tokens = re.split(r'[,\s\n]+', text)
            values = [float(t) for t in tokens if t]
            if len(values) < 10:
                raise ValueError(f"Manual input requires at least 10 numbers (you entered {len(values)}).")
            if len(values) > 170:
                raise ValueError(f"Manual input cannot exceed 170 numbers (you entered {len(values)}).")
            values = np.array(values, dtype=np.float64)
        
        elif source == 'Upload File (CSV/Excel)':
            if not self.uploaded_file_path:
                raise ValueError("No file uploaded. Please select a CSV or Excel file.")
            if self.uploaded_file_path.endswith('.csv'):
                df = pd.read_csv(self.uploaded_file_path)
            else:
                df = pd.read_excel(self.uploaded_file_path)
            
            selected_col = self.column_combo.get()
            if selected_col and selected_col in df.columns:
                values = pd.to_numeric(df[selected_col], errors='coerce').dropna().values
            else:
                numeric_df = df.select_dtypes(include=[np.number])
                values = numeric_df.iloc[:, 0].dropna().values if not numeric_df.empty else np.array([])
                
            if len(values) < 10:
                raise ValueError(f"Selected column requires at least 10 valid numbers (found {len(values)}).")
            if len(values) > 170:
                raise ValueError(f"Selected column cannot exceed 170 valid numbers (found {len(values)}).")
            values = np.array(values, dtype=np.float64)

        if self.center_mean_var.get() and len(values) > 0:
            values = values - np.mean(values)

        return values

    def start_simulation_thread(self):
        try:
            values = self.get_input_values()
        except Exception as e:
            messagebox.showerror("Input Error", str(e))
            return
            
        target_scales = []
        for s, var in self.scale_vars.items():
            if var.get():
                target_scales.append(s)
                
        use_exact_all = False
        if self.all_scale_var.get() and self.all_scale_val is not None:
            if len(values) in (10, 11, 12):
                target_scales.append(self.all_scale_val)
                use_exact_all = True
            else:
                messagebox.showwarning("Warning", "The 'All' exact permutations option is only available for Sample Sizes of 10, 11, or 12.")
                return
            
        if not target_scales:
            messagebox.showwarning("Warning", "Please select at least one scale (n) or 'All'.")
            return
            
        target_scales = sorted(list(set(target_scales)))
        
        # Parse selected batch size
        batch_map = {'1M': 1_000_000, '2M': 2_000_000, '5M': 5_000_000, '10M': 10_000_000}
        selected_batch_str = self.batch_size_combo.get()
        batch_size_val = batch_map.get(selected_batch_str, 2_000_000)
        
        sim_seed_mode = self.sim_seed_mode_combo.get()
        if sim_seed_mode == 'Manual Seed':
            sim_seed = int(self.sim_seed_entry.get()) if self.sim_seed_entry.get().strip().isdigit() else 1
        else:
            sim_seed = int(self.sim_seed_entry.get()) if self.sim_seed_entry.get().strip().isdigit() else None
            
        csv_filename = self.csv_name_entry.get().strip() or "dw_results.csv"
        excel_filename = self.excel_name_entry.get().strip() or "dw_results.xlsx"
        img_filename = self.img_name_entry.get().strip() or "dw_boxplots.png"
        
        if self.output_directory:
            csv_filename = os.path.join(self.output_directory, csv_filename)
            excel_filename = os.path.join(self.output_directory, excel_filename)
            img_filename = os.path.join(self.output_directory, img_filename)
            
        data_source_name = f"{self.data_source_combo.get()} (Col: {self.column_combo.get()})" if self.data_source_combo.get() == 'Upload File (CSV/Excel)' else self.data_source_combo.get()
        sample_seed_val = self.seed_entry.get().strip() or "N/A"
        sim_seed_val = self.sim_seed_entry.get().strip() or "N/A"
        sim_seed_mode_val = self.sim_seed_mode_combo.get()
        
        self.run_btn.configure(state="disabled")
        self.stop_btn.configure(state="normal")
        self.reset_btn.configure(state="disabled")
        self.stop_requested = False
        self.progress_bar.set(0)
        
        def run_thread():
            try:
                history, rows, headers, elapsed = run_simulation(
                    values, target_scales, batch_size_val,
                    self.progress_bar, self.eta_label, 
                    csv_filename, excel_filename, data_source_name, 
                    sample_seed_val, sim_seed_mode_val, sim_seed_val,
                    self.update_gui_progress, self.check_stop, 
                    sim_seed=sim_seed, use_exact_all=use_exact_all
                )
                self.after(0, lambda: self.on_simulation_complete(history, rows, headers, elapsed, img_filename))
            except InterruptedError:
                self.after(0, self.on_simulation_stopped)
            except Exception as e:
                self.after(0, lambda: self.on_simulation_error(str(e)))
                
        threading.Thread(target=run_thread, daemon=True).start()

    def update_gui_progress(self, progress, status_text):
        self.after(0, lambda: self.progress_bar.set(progress))
        self.after(0, lambda: self.eta_label.configure(text=f"Status: {status_text}"))

    def check_stop(self):
        return self.stop_requested

    def on_simulation_stopped(self):
        self.run_btn.configure(state="normal")
        self.stop_btn.configure(state="disabled")
        self.reset_btn.configure(state="normal")
        self.progress_bar.set(0)
        self.eta_label.configure(text="Status: Simulation stopped by user.")
        messagebox.showinfo("Stopped", "Simulation was interrupted.")

    def on_simulation_error(self, err_msg):
        self.run_btn.configure(state="normal")
        self.stop_btn.configure(state="disabled")
        self.reset_btn.configure(state="normal")
        self.progress_bar.set(0)
        self.eta_label.configure(text="Status: Error encountered.")
        messagebox.showerror("Simulation Error", f"An error occurred:\n{err_msg}")

    def on_simulation_complete(self, history, rows, headers, elapsed, img_filename):
        self.run_btn.configure(state="normal")
        self.stop_btn.configure(state="disabled")
        self.reset_btn.configure(state="normal")
        self.eta_label.configure(text=f"Status: Completed successfully in {elapsed:.1f}s")
        
        table_str = tabulate(rows, headers=headers, tablefmt="fancy_grid")
        self.output_textbox.delete("0.1", "end")
        self.output_textbox.insert("0.1", table_str)
        
        self.render_boxplots(history, img_filename)

    def render_boxplots(self, history, img_filename):
        if self.canvas_widget:
            self.canvas_widget.get_tk_widget().destroy()
            self.canvas_widget = None
            
        fig, ax = plt.subplots(figsize=(18, 9), dpi=200)
        scales = list(history.keys())
        data_to_plot = [history[s] for s in scales]
        
        try:
            sample_size = len(self.get_input_values())
        except Exception:
            sample_size = len(next(iter(history.values()))) if history else 0
            
        labels = [("All" if s == math.factorial(sample_size) else (f"{s:,}" if s < 1_000_000 else (f"{s//1_000_000}M" if s < 1_000_000_000 else f"{s//1_000_000_000}B"))) for s in scales]
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, showmeans=True,
                        meanprops={"marker":"D", "markerfacecolor":"green", "markeredgecolor":"green", "markersize":6},
                        medianprops={"color":"red", "linewidth":1.5})
                        
        for patch in bp['boxes']:
            patch.set_facecolor(self.boxplot_color)
            patch.set_linewidth(1.0)
            
        for whisker in bp['whiskers']:
            whisker.set_linewidth(1.0)
        for cap in bp['caps']:
            cap.set_linewidth(1.0)
        for flier in bp['fliers']:
            flier.set(marker='o', color='black', alpha=0.5, markersize=4)
            
        ax.axhline(2.0, color='red', linestyle='--', linewidth=1.2, label='d = 2 (No Autocorrelation)')
        
        ax.set_title(f"Durbin-Watson Distribution across Scales (Sample Size: {sample_size})", fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel("Simulation Scale (n)", fontsize=14, fontweight='bold', labelpad=10)
        ax.set_ylabel("Durbin-Watson Statistic", fontsize=14, fontweight='bold', labelpad=10)
        
        ax.tick_params(axis='both', labelsize=12)
        
        ax.grid(True, linestyle='--', alpha=0.5, linewidth=0.8)
        
        legend_elements = [
            Line2D([0], [0], color='red', lw=1.5, label='Median'),
            Line2D([0], [0], marker='D', color='w', markerfacecolor='green', markeredgecolor='green', markersize=7, label='Mean'),
            Line2D([0], [0], color='red', linestyle='--', lw=1.2, label='d = 2 (No Autocorrelation)')
        ]
        ax.legend(handles=legend_elements, loc='upper left', frameon=True, fontsize=11)
        
        sample_seed_val = self.seed_entry.get().strip() or "N/A"
        sim_seed_val = self.sim_seed_entry.get().strip() or "N/A"
        
        if self.center_mean_var.get():
            mean_text = "Sample mean = zero"
        else:
            try:
                current_vals = self.get_input_values()
                curr_mean = np.mean(current_vals)
                mean_text = f"Sample mean = {curr_mean:.5f}"
            except Exception:
                mean_text = "Sample mean = N/A"
        
        seed_text = f"Sample seed = {sample_seed_val}\nSimulation seed = {sim_seed_val}\n{mean_text}"
        
        ax.text(0.02, 0.03, seed_text, transform=ax.transAxes, fontsize=11,
                verticalalignment='bottom', horizontalalignment='left',
                bbox=dict(boxstyle='round,pad=0.6', facecolor='white', alpha=0.85, edgecolor='gray', linewidth=0.8))
        
        fig.tight_layout()
        fig.savefig(img_filename, dpi=400)
        plt.close(fig)
        
        self.canvas_widget = FigureCanvasTkAgg(fig, master=self.plot_frame)
        self.canvas_widget.draw()
        self.canvas_widget.get_tk_widget().pack(side="top", fill="x", expand=True, padx=10, pady=10)

if __name__ == "__main__":
    app = DurbinWatsonApp()
    app.mainloop()